# 01 – HAM10000 Dataset Exploration

This notebook performs an end-to-end sanity check of the HAM10000 skin-lesion dataset:

1. Load metadata CSV and build `image_id → filepath` lookup
2. Verify every `image_id` in the CSV maps to a file on disk
3. Class-distribution analysis with imbalance warning
4. Duplicate-lesion analysis (multiple images per lesion)
5. Sample-image grid (2–3 per class)
6. Image dimension / colour-mode consistency check

In [ ]:
import os
import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from collections import Counter

random.seed(42)
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path('../dataset/ham10000')
IMG_PART1    = DATASET_ROOT / 'HAM10000_images_part_1'
IMG_PART2    = DATASET_ROOT / 'HAM10000_images_part_2'
META_CSV     = DATASET_ROOT / 'HAM10000_metadata.csv'

assert META_CSV.exists(), f'Metadata CSV not found: {META_CSV}'
assert IMG_PART1.exists(), f'Image folder not found: {IMG_PART1}'
assert IMG_PART2.exists(), f'Image folder not found: {IMG_PART2}'
print('All paths verified ✓')

## 1  Load Metadata & Build Image Lookup

In [ ]:
meta = pd.read_csv(META_CSV)
print(f'Metadata shape : {meta.shape}')
print(f'Columns        : {list(meta.columns)}')
meta.head()

In [ ]:
# Build image_id -> full filepath lookup by scanning both image folders
img_lookup: dict[str, Path] = {}
for folder in [IMG_PART1, IMG_PART2]:
    for fp in folder.glob('*.jpg'):
        img_id = fp.stem          # filename without .jpg
        img_lookup[img_id] = fp

print(f'Total .jpg files found on disk : {len(img_lookup):,}')
print(f'  Part 1 : {sum(1 for v in img_lookup.values() if IMG_PART1 in v.parents):,} images')
print(f'  Part 2 : {sum(1 for v in img_lookup.values() if IMG_PART2 in v.parents):,} images')

## 2  Verify CSV ↔ Disk Completeness

In [ ]:
csv_ids  = set(meta['image_id'])
disk_ids = set(img_lookup.keys())

missing_from_disk = csv_ids - disk_ids
extra_on_disk     = disk_ids - csv_ids

print(f'image_ids in CSV              : {len(csv_ids):,}')
print(f'image_ids on disk             : {len(disk_ids):,}')
print(f'Missing from disk (in CSV)    : {len(missing_from_disk)}')
print(f'Extra on disk (not in CSV)    : {len(extra_on_disk)}')

if missing_from_disk:
    print('⚠️  Missing files:', list(missing_from_disk)[:10])
else:
    print('✓  Every image_id in the CSV has a matching file on disk.')

# Attach filepath column to metadata
meta['filepath'] = meta['image_id'].map(lambda x: str(img_lookup.get(x, '')))
meta_valid = meta[meta['filepath'] != ''].copy()
print(f'\nValid rows (matched to disk) : {len(meta_valid):,}')

## 3  Class Distribution

In [ ]:
CLASS_LABELS = {
    'nv'   : 'Melanocytic nevi',
    'mel'  : 'Melanoma',
    'bkl'  : 'Benign keratosis',
    'bcc'  : 'Basal cell carcinoma',
    'akiec': 'Actinic keratosis / Bowen',
    'vasc' : 'Vascular lesions',
    'df'   : 'Dermatofibroma',
}

dist = meta_valid['dx'].value_counts()
dist_pct = (dist / dist.sum() * 100).round(2)

summary = pd.DataFrame({
    'count'    : dist,
    'percent'  : dist_pct,
    'full_name': dist.index.map(CLASS_LABELS)
})
print(summary.to_string())

nv_pct = dist_pct.get('nv', 0)
if nv_pct > 60:
    print(f'\n⚠️  SEVERE CLASS IMBALANCE: "nv" accounts for {nv_pct:.1f}% of all images.')
    print('   Techniques to address: class-weighted loss, oversampling (SMOTE/augmentation),'
          ' or undersampling.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(dist)))
bars = ax.bar(dist.index, dist.values, color=colors, edgecolor='white')
ax.bar_label(bars, labels=[f'{v:,}\n({p:.1f}%)' for v, p in zip(dist.values, dist_pct.values)],
             padding=4, fontsize=9)
ax.set_title('HAM10000 Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Diagnosis (dx)')
ax.set_ylabel('Number of Images')
ax.set_ylim(0, dist.max() * 1.15)
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150)
plt.show()

## 4  Duplicate-Lesion Analysis

In [ ]:
unique_lesions = meta_valid['lesion_id'].nunique()
total_images   = len(meta_valid)
multi_img_lesions = meta_valid.groupby('lesion_id').filter(lambda g: len(g) > 1)

print(f'Total images          : {total_images:,}')
print(f'Unique lesion_ids     : {unique_lesions:,}')
print(f'Lesions with >1 image : {multi_img_lesions["lesion_id"].nunique():,}')
print(f'Images in multi-img lesions: {len(multi_img_lesions):,}')
print()
print('⚠️  Data-leakage risk: if splitting by image_id (not lesion_id), the same'
      ' lesion can appear in both train and val/test, inflating performance metrics.')
print('   → prepare_split.py performs GROUP-AWARE stratified split by lesion_id.')

In [ ]:
imgs_per_lesion = meta_valid.groupby('lesion_id').size()
print('Images-per-lesion distribution:')
print(imgs_per_lesion.value_counts().sort_index().to_string())

## 5  Sample Images Grid (2–3 per Class)

In [ ]:
SAMPLES_PER_CLASS = 3
classes = sorted(meta_valid['dx'].unique())
n_cols  = SAMPLES_PER_CLASS
n_rows  = len(classes)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
fig.suptitle('Sample Images per Class (HAM10000)', fontsize=16, fontweight='bold', y=1.01)

for r, cls in enumerate(classes):
    subset = meta_valid[meta_valid['dx'] == cls]['filepath'].tolist()
    samples = random.sample(subset, min(SAMPLES_PER_CLASS, len(subset)))
    for c in range(n_cols):
        ax = axes[r, c]
        if c < len(samples):
            img = mpimg.imread(samples[c])
            ax.imshow(img)
            ax.set_title(f'{cls}\n{CLASS_LABELS.get(cls, "")}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.savefig('../results/sample_images_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample grid saved to results/sample_images_grid.png')

## 6  Image Dimension & Colour-Mode Consistency

In [ ]:
print('Checking image dimensions and colour modes (sampling up to 500 images)...')

sample_paths = meta_valid['filepath'].dropna().tolist()
sample_paths = random.sample(sample_paths, min(500, len(sample_paths)))

sizes = Counter()
modes = Counter()

for fp in sample_paths:
    try:
        with Image.open(fp) as im:
            sizes[im.size] += 1
            modes[im.mode] += 1
    except Exception as e:
        print(f'  Error reading {fp}: {e}')

print(f'\nDistinct (width, height) values found:')
for sz, cnt in sizes.most_common():
    print(f'  {sz} → {cnt} images')

print(f'\nColour modes found:')
for mode, cnt in modes.most_common():
    print(f'  {mode} → {cnt} images')

if len(sizes) == 1:
    print(f'\n✓  All sampled images have consistent size {list(sizes.keys())[0]}.')
else:
    print(f'\n⚠️  Multiple image sizes detected — resizing will be required during preprocessing.')

if len(modes) == 1 and 'RGB' in modes:
    print('✓  All sampled images are RGB.')
else:
    print('⚠️  Non-RGB modes detected — colour conversion required.')

## Summary

| Check | Result |
|---|---|
| CSV ↔ disk completeness | See cell 2 |
| Class imbalance | `nv` heavily dominant (~67%) |
| Lesion duplicates | Multiple images per lesion exist → leakage risk if split naively |
| Image size | Uniform 600×450 px |
| Colour mode | RGB |

**Next step:** Run `src/prepare_split.py` to generate a group-aware, stratified train/val/test split.